# QEvict completion ablation for Reviewer 2 (Colab, A100)

> *How sensitive is QEvict's model quality to the accuracy of the calibrated window-level
> completion term for windows that are not selected for token-level attention?*

**Model:** Llama-3.1-8B-Instruct. **Operating point:** `utils.config.OPERATING_POINT`, i.e. budget 0.20,
W = 8, `quant_ratio` 0.70 in `bytes` mode, read ratio ρ = 0.25, int2, flash backend, `bidir`/`dequant`, greedy.

| Experiment | Question | Arms |
|---|---|---|
| **1** Matched-state accuracy | How close is the completion term to the true unread-window mass? | collector on the shipped method |
| **2** Is completion needed? | calibrated vs off | `cal`, `off`, ceiling ρ = 1.0, stress ρ = 0.10 |
| **3** Is calibration needed? | calibrated vs uncalibrated (`ℓ̂ − log Z_A`) | `cal`, `uncal` |
| **4** Sensitivity | accuracy vs controlled mass error | scale `c`, total-preserving noise `σ`, coupled |

`cal` (the shipped method) is the control arm in every experiment; `cal_repeat` measures the run-to-run noise floor.

### What this branch can run today (audited at `2b860f2`)

| Piece | Status |
|---|---|
| `cal`, `cal_repeat`, ceiling ρ = 1.0, `cal` at ρ = 0.10 | **runnable** |
| Exp 1a: fill bias and spread against the read windows, unread mass, read-gate output error (existing collector arrays) | **runnable** |
| Exp 1b: per-variant counterfactual outputs, mass oracle, `Z_A`-normalised `m*(u)` | **not on the branch** |
| `quant_completion` knob (`off` / `uncalibrated` / scale / noise / coupled) | **not on the branch**, so the Exp 2–4 variant arms are **blocked** |

The readiness cell detects the knob on whatever commit gets cloned. Blocked arms are **refused, never run**: this repo's
config loader silently ignores unknown keys, so overriding a missing knob would run the control and file it under the
variant's name. Every result must also carry `read_gate == gated` and echo its knobs in the sidecar, or it is moved to
`refused/`.

**Planned knob contract** (what the arms below pass once the knob exists):
`cache.quant_completion ∈ {calibrated, off, uncalibrated}` (not `none`: the override parser turns `none` into Python `None`),
`cache.quant_completion_scale` (float, 1.0), `cache.quant_completion_sigma` (nats, 0.0; total-preserving, persistent per
layer × query head × window), `cache.quant_completion_seed` (int), `cache.quant_completion_scores ∈ {output, coupled}`.
All of them act after selection and after `δ`.

### How to use
1. Runtime → Change runtime type → **A100**. High-RAM helps Experiment 1.
2. Colab secrets (key icon): `HF_TOKEN` with access to `meta-llama/Llama-3.1-8B-Instruct`; `GITHUB_TOKEN` if the repo is private.
3. Run top to bottom. Results go to Drive and **every job resumes**, so after a disconnect, run everything again.

### Rough time on one A100

Arithmetic, not measured on Colab. It assumes ~55 ms/token TPOT at B=1 (`DISTANCE_TO_GOAL.md` §1) plus prefill, at the
LongBench output caps (128 / 32 / 512 / 32 tokens).

| Block | Jobs | Time |
|---|---|---|
| Setup (install, model, data, preflight tests) | – | 15–30 min |
| Experiment 1a, 4 tasks × `EXP1_PROMPTS` = 8 | 4 | unmeasured; budget 30–60 min per task (full-KV shadow attention every step) |
| One LongBench arm, 4 tasks × 100 examples | 4 | ~1–1.5 h (QMSum is about two thirds of it) |
| Tier 1, runnable today: `cal`, `cal_repeat`, ceiling | 12 | ~4 h |
| Tier 1 once the knob lands: + `off`, `uncal`, `scale0.5`, `scale2`, 3 × `noise1x` | +28 | +~9 h |

Colab ends sessions after roughly 12–24 h. Rerun the notebook and finished jobs are skipped.

## 0. Settings

In [ ]:
# ===================== Settings: edit here =====================
BRANCH       = "Clustered_int2_paused_fast"
REPO_URL     = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT   = None      # e.g. "2b860f2". None = branch head. The SHA is recorded either way.
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_DIR    = "/content/models/Llama-3.1-8B-Instruct"   # the name must contain "instruct": the chat-template gate reads it
DRIVE_ROOT   = "/content/drive/MyDrive/recoverkv_completion_ablation"

TASKS        = ["narrativeqa", "musique", "qmsum", "triviaqa"]
NUM_SAMPLES  = 100       # per task; "max" = the full split (200)
TIER         = 1         # 1 = the minimum set that answers the reviewer; 2 = adds the rest
RUN_GSM8K    = False     # tier-2 task; only runs when TIER >= 2

# Experiment 1: collector on the shipped method
EXP1_PROMPTS = 8
EXP1_PREFILL = 4096
EXP1_GEN     = 256

BOOTSTRAP_ITERS        = 10_000
RUN_PREFLIGHT_TESTS    = True
FLASH_ATTN_FROM_SOURCE = False   # only if no prebuilt wheel matches (slow: ~1-2 h)
ALLOW_NON_A100         = False

# ===================== Fixed: these define the experiment =====================
OPERATING_POINT = dict(cache_budget=0.20, window_size=8, quant_ratio=0.70,
                       quant_budget_mode="bytes", quant_promotion="bidir",
                       quant_promote_source="dequant")
GATE = 0.25
KNOB_FIELDS = ("quant_completion", "quant_completion_scale", "quant_completion_sigma",
               "quant_completion_seed", "quant_completion_scores")

## 1. GPU, Drive, repository

In [ ]:
import subprocess, json, os, sys, shutil, time, math, glob, zipfile, io, re, warnings
from pathlib import Path

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100.")
gpu_name, mem_mib, cc = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = float(mem_mib) / 1024
print(f"GPU: {gpu_name}  {GPU_MEM_GB:.0f} GB  sm_{cc.replace('.', '')}")
if cc != "8.0" and not ALLOW_NON_A100:
    raise RuntimeError(f"Expected an A100 (sm_80), got {gpu_name} (sm_{cc}). The kernels were "
                       "compile-checked for sm80. Set ALLOW_NON_A100 = True to override.")
SKIP_OOM = GPU_MEM_GB < 70
print(f"skip_oom = {SKIP_OOM}" + ("  (40 GB card: the longest NarrativeQA prompts may OOM; they score 0 "
      "THUDM-style and are dropped from the paired comparisons)" if SKIP_OOM else ""))

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
for d in ("longbench", "gsm8k", "exp1", "logs", "refused", "results"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

def _secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

REPO = Path("/content/RecoverKv")
gh = _secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never echo the tokenised URL
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(gh or "\0", "***"))
    return r.stdout.strip()

if not REPO.exists():
    _git("clone", "--branch", BRANCH, url, str(REPO))
else:
    _git("fetch", "origin", BRANCH, cwd=REPO)
    _git("checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO)
if PIN_COMMIT:
    _git("checkout", PIN_COMMIT, cwd=REPO)
SHA = _git("rev-parse", "HEAD", cwd=REPO)
print("commit", SHA)

# Quality rows do not compare across commits, so one Drive root holds one commit.
manifest = ROOT / "manifest.json"
if manifest.exists():
    m = json.loads(manifest.read_text())
    if m["sha"] != SHA:
        raise RuntimeError(f"{ROOT} holds results from commit {m['sha'][:10]}, and this clone is "
                           f"{SHA[:10]}. Set PIN_COMMIT = '{m['sha'][:10]}' to resume, or point "
                           "DRIVE_ROOT at a new folder.")
else:
    manifest.write_text(json.dumps({"sha": SHA, "branch": BRANCH, "model": MODEL_ID,
                                    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
                                   indent=2))

## 2. Install the pinned stack

* torch 2.7.1 gives Triton 3.3.1, the version the kernels were compile-checked on.
* transformers is pinned to 4.47.1: the windowed cache refuses anything newer.
* `datasets` stays below 3, as `environment.yml` pins it.

The notebook kernel never imports torch: every job runs as a subprocess, so no runtime restart is needed.

In [ ]:
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)

pip("torch==2.7.1", "torchvision==0.22.1", "torchaudio==2.7.1",
    "--index-url", "https://download.pytorch.org/whl/cu126")
pip("transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
    "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm", "pandas", "matplotlib",
    "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba", "pytest")

# flash-attn: a prebuilt wheel matching this torch / python / C++ ABI
tv, abi, py = subprocess.check_output([sys.executable, "-c",
    "import torch, sys; print(torch.__version__.split('+')[0]); "
    "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
    "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], text=True).split()
tmm = ".".join(tv.split(".")[:2])
import urllib.request
def _exists(u):
    try:
        req = urllib.request.Request(u, headers={"Range": "bytes=0-0"})
        with urllib.request.urlopen(req, timeout=30) as r:
            return r.status in (200, 206)
    except Exception:
        return False
wheel = None
for v in ("2.8.3", "2.8.2", "2.8.1", "2.8.0.post2", "2.7.4.post1"):
    u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
         f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
    if _exists(u):
        wheel = u
        break
if wheel:
    print("flash-attn wheel:", wheel.rsplit("/", 1)[1])
    pip(wheel)
elif FLASH_ATTN_FROM_SOURCE:
    subprocess.run([sys.executable, "-m", "pip", "install", "flash-attn==2.8.3", "--no-build-isolation"],
                   check=True, env=dict(os.environ, MAX_JOBS="8"))
else:
    raise RuntimeError(f"No prebuilt flash-attn wheel for torch {tv} / {py} / cxx11abi={abi}. "
                       "Set FLASH_ATTN_FROM_SOURCE = True (about 1-2 h) and rerun this cell.")

check = r"""
import torch, triton, transformers, flash_attn, datasets
print("torch", torch.__version__, "| triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__, "| datasets", datasets.__version__)
assert torch.cuda.is_available()
from flash_attn import flash_attn_func
x = torch.randn(1, 256, 8, 128, device="cuda", dtype=torch.float16)
flash_attn_func(x, x, x, causal=True); torch.cuda.synchronize()
assert triton.__version__.startswith("3.3"), "the kernels were compile-checked on Triton 3.3.1"
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()
print("environment ok")
"""
subprocess.run([sys.executable, "-c", check], cwd=REPO, check=True)

## 3. Model and data

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
HF_TOKEN = _secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add a Colab secret HF_TOKEN with access to meta-llama/Llama-3.1-8B-Instruct.")
assert "instruct" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'instruct'"
snapshot_download(MODEL_ID, local_dir=MODEL_DIR, token=HF_TOKEN,
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])

LB_DIR = REPO / "data" / "longbench"
LB_DIR.mkdir(parents=True, exist_ok=True)
need = [t for t in TASKS if not (LB_DIR / f"{t}.jsonl").exists()]
if need:
    zp = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        for t in need:
            member = next(n for n in names if n == f"{t}.jsonl" or n.endswith(f"/{t}.jsonl"))
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
for t in TASKS:
    print(f"{t:12s} {sum(1 for _ in open(LB_DIR / f'{t}.jsonl'))} examples")

ENV = dict(os.environ, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", LONGBENCH_LOCAL_DIR=str(LB_DIR),
           HF_TOKEN=HF_TOKEN, PYTHONUNBUFFERED="1")

if RUN_GSM8K and TIER >= 2 and not (REPO / "data/gsm8k_cot").exists():
    subprocess.run([sys.executable, "-m", "data.gsm8k_loader", "--out", "data/gsm8k_cot"],
                   cwd=REPO, env=ENV, check=True)

## 4. Readiness check

Checks four things on the clone:
* the gate is wired, i.e. `fused_gate` has non-test callers (the `CLAUDE.md` check);
* every config is at the operating point;
* the completion knob is present on `CacheConfig`;
* the collector has completion rungs.

It also runs the gate and fill tests and the sm80 compile tests on this GPU.

In [ ]:
probe = r"""
import json, dataclasses, subprocess, pathlib
from utils.config import CacheConfig
fields = {f.name for f in dataclasses.fields(CacheConfig)}
callers = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'",
                         shell=True, capture_output=True, text=True).stdout.splitlines()
src = pathlib.Path("modules/evaluation/observation_collector.py").read_text()
print(json.dumps({
    "knobs": {k: k in fields for k in %r},
    "gate_callers": [c for c in callers if "def fused_gate" not in c],
    "exp1b_rungs": "--completion-rungs" in src,
}))
""" % (KNOB_FIELDS,)
info = json.loads(subprocess.check_output([sys.executable, "-c", probe], cwd=REPO, env=ENV, text=True))
op = subprocess.run([sys.executable, "scripts/check_operating_point.py"], cwd=REPO, env=ENV,
                    capture_output=True, text=True)

READY = {
    "gate_wired": bool(info["gate_callers"]),
    "operating_point": op.returncode == 0,
    "knob": all(info["knobs"].values()),
    "exp1b": info["exp1b_rungs"],
}
if not READY["gate_wired"]:
    raise RuntimeError("fused_gate has no non-test callers: the gate is not in the decode step. Stop.")
if not READY["operating_point"]:
    print(op.stdout[-3000:])
    raise RuntimeError("A config diverges from the operating point (scripts/check_operating_point.py).")

if RUN_PREFLIGHT_TESTS:
    t = subprocess.run([sys.executable, "-m", "pytest", "-q", "-x",
                        "tests/test_gate_fill_calibration.py", "tests/test_gate_union_is_per_head.py",
                        "tests/test_kernel_compiles.py"], cwd=REPO, env=ENV, capture_output=True, text=True)
    print(t.stdout[-1500:])
    if t.returncode != 0:
        raise RuntimeError("Preflight tests failed on this GPU. See the output above.")

print(f"\ncommit {SHA[:10]}")
print(f"  gate wired (non-test callers)     {READY['gate_wired']}")
print(f"  operating point                   {READY['operating_point']}")
print(f"  completion knob                   {READY['knob']}   {info['knobs']}")
print(f"  collector completion rungs (1b)   {READY['exp1b']}")
print("\nRunnable now: cal, cal_repeat, ceiling, stress cal, Experiment 1a"
      + ("" if READY["knob"] else "\nBLOCKED until the knob lands: off, uncal, scale*, noise*, coupled*, stress off"))

## 5. Job runner (resumable, verified, refuses mislabelled runs)

In [ ]:
LOGS = ROOT / "logs"
RUNS = Path("/content/runs")

class Blocked(Exception):
    pass

def run(cmd, log_name):
    cmd = [str(c) for c in cmd]
    log = LOGS / f"{log_name}.log"
    print("$", " ".join(cmd))
    t0, tail = time.time(), []
    keep = ("examples done", "read gate", "OOM", "Error", "error:", "NOT-GATED", "Traceback", "score=")
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-60:]
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:200])
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{log_name} failed (rc={rc}). Full log: {log}")
    print(f"    finished in {(time.time() - t0) / 60:.1f} min")

# ---- arms --------------------------------------------------------------------
CTRL = {"mode": "calibrated", "scale": 1.0, "sigma": 0.0, "seed": 0, "scores": "output"}
def C(**kw):
    return {**CTRL, **kw}

ARMS = {}
def arm(name, tier, exp, gate=GATE, comp=None, sigma_mult=None):
    ARMS[name] = dict(tier=tier, exp=exp, gate=gate, comp=comp, sigma_mult=sigma_mult)

arm("cal",          1, "control")
arm("cal_repeat",   1, "control")                       # run-to-run noise floor
arm("ceiling_g1.0", 1, "2", gate=1.0)                   # nothing unread: perfect completion
arm("off",          1, "2", comp=C(mode="off"))
arm("uncal",        1, "3", comp=C(mode="uncalibrated"))
arm("scale0.5",     1, "4A", comp=C(scale=0.5))
arm("scale2",       1, "4A", comp=C(scale=2.0))
for s in range(3):
    arm(f"noise1x_s{s}", 1, "4B", comp=C(seed=s), sigma_mult=1)
# tier 2
for c in (0.8, 1.25, 4.0):
    arm(f"scale{c:g}", 2, "4A", comp=C(scale=c))
for mult in (2, 4):
    for s in range(3):
        arm(f"noise{mult}x_s{s}", 2, "4B", comp=C(seed=s), sigma_mult=mult)
arm("stress_g0.10_cal", 2, "2", gate=0.10)
arm("stress_g0.10_off", 2, "2", gate=0.10, comp=C(mode="off"))
for c in (0.5, 2.0):
    arm(f"coupled_scale{c:g}", 2, "4C", comp=C(scale=c, scores="coupled"))

SIGMA_FILE = ROOT / "exp1" / "sigma_hat.json"

def resolved_comp(a):
    """The completion settings an arm must run with, or None for 'whatever the commit ships'."""
    if a["comp"] is None:
        return dict(CTRL) if READY["knob"] else None
    if not READY["knob"]:
        raise Blocked("the quant_completion knob is not on this commit")
    c = dict(a["comp"])
    if a["sigma_mult"]:
        if not SIGMA_FILE.exists():
            raise Blocked("noise arms need sigma_hat_U from Experiment 1a. Run section 6 first")
        c["sigma"] = a["sigma_mult"] * json.loads(SIGMA_FILE.read_text())["sigma_hat_U"]
    return c

def overrides(a, comp):
    ov = [f"model.name={MODEL_DIR}", f"cache.quant_gate_ratio={a['gate']}"]
    if comp is not None:
        ov += [f"cache.quant_completion={comp['mode']}", f"cache.quant_completion_scale={comp['scale']}",
               f"cache.quant_completion_sigma={comp['sigma']}", f"cache.quant_completion_seed={comp['seed']}",
               f"cache.quant_completion_scores={comp['scores']}"]
    return ov

def verify(meta, a, comp):
    """Everything a sidecar must say for its number to belong to this arm."""
    errs = []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != "gated":
        errs.append(f"read_gate verdict {rg.get('verdict')!r}, need 'gated'")
    rf = rg.get("read_fraction")
    if rf is None or not (a["gate"] - 1e-6 <= rf <= min(1.0, a["gate"] + 0.15) + 1e-6):
        errs.append(f"realised read fraction {rf} for requested ratio {a['gate']}")
    for k, v in OPERATING_POINT.items():
        if k in meta:
            mv = meta[k]
            bad = abs(float(mv) - v) > 1e-9 if isinstance(v, float) else mv != v
            if bad:
                errs.append(f"{k} = {mv!r}, expected {v!r}")
    if comp is not None:
        got = meta.get("quant_completion")
        if not isinstance(got, dict):
            errs.append("no quant_completion record in the sidecar: the knob did not reach the run")
        else:
            for k, v in comp.items():
                g = got.get(k)
                ok = (g is not None and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))) \
                     if isinstance(v, (int, float)) else g == v
                if not ok:
                    errs.append(f"quant_completion.{k} = {g!r}, expected {v!r}")
    return errs

def _refuse(files, tag, errs):
    dst = ROOT / "refused" / tag
    dst.mkdir(parents=True, exist_ok=True)
    for f in files:
        if Path(f).exists():
            shutil.move(str(f), dst / Path(f).name)
    raise RuntimeError(f"REFUSED {tag}:\n  " + "\n  ".join(errs) + f"\n  files moved to {dst}")

# ---- LongBench -----------------------------------------------------------------
def lb_job(name, task):
    a = ARMS[name]
    comp = resolved_comp(a)                     # raises Blocked
    dst = ROOT / "longbench" / name
    dst.mkdir(parents=True, exist_ok=True)
    meta_f, pred_f = dst / f"{task}.meta.json", dst / f"{task}.jsonl"
    if meta_f.exists() and pred_f.exists():     # the sidecar is written last, so it is the done-marker
        errs = verify(json.loads(meta_f.read_text()), a, comp)
        if errs:
            raise RuntimeError(f"STALE {name}/{task} on Drive does not match this arm:\n  "
                               + "\n  ".join(errs) + "\n  Move it away to rerun.")
        return "done"
    local = RUNS / "longbench" / name
    local.mkdir(parents=True, exist_ok=True)
    for f in (local / f"{task}.jsonl", local / f"{task}.meta.json"):
        f.unlink(missing_ok=True)               # the runner's resume would trust a partial jsonl
    cmd = [sys.executable, "main.py", "--config", "configs/longbench_ours_flash_attn.yaml", "--override",
           f"longbench.datasets=[{task}]", f"longbench.output_dir={local}",
           f"longbench.num_samples={NUM_SAMPLES}", "longbench.resume=false",
           f"longbench.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)]
    run(cmd, f"lb_{name}_{task}")
    meta = json.loads((local / f"{task}.meta.json").read_text())
    errs = verify(meta, a, comp)
    if errs:
        _refuse([local / f"{task}.jsonl", local / f"{task}.meta.json"], f"{name}_{task}", errs)
    shutil.copy(local / f"{task}.jsonl", pred_f)
    shutil.copy(local / f"{task}.meta.json", meta_f)
    return "ran"

# ---- GSM8K ---------------------------------------------------------------------
GSM8K_ARMS = ["cal", "cal_repeat", "off", "uncal", "scale0.5", "scale2"]

def gsm8k_job(name):
    a = ARMS[name]
    comp = resolved_comp(a)
    dst = ROOT / "gsm8k" / name
    if (dst / "meta.json").exists() and (dst / "predictions.jsonl").exists():
        errs = verify(json.loads((dst / "meta.json").read_text()), a, comp)
        if errs:
            raise RuntimeError(f"STALE gsm8k/{name}:\n  " + "\n  ".join(errs))
        return "done"
    local = RUNS / "gsm8k" / name
    shutil.rmtree(local, ignore_errors=True)
    cmd = [sys.executable, "main.py", "--config", "configs/gsm8k_budget20.yaml", "--override",
           f"gsm8k.output_dir={local}", f"gsm8k.skip_oom={str(SKIP_OOM).lower()}", *overrides(a, comp)]
    run(cmd, f"gsm8k_{name}")
    meta = json.loads((local / "meta.json").read_text())
    errs = verify(meta, a, comp)
    if errs:
        _refuse([local / "predictions.jsonl", local / "meta.json"], f"gsm8k_{name}", errs)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(local / "predictions.jsonl", dst / "predictions.jsonl")
    shutil.copy(local / "meta.json", dst / "meta.json")
    return "ran"

def run_grid(tier_max=TIER, tasks=TASKS, only=None):
    names = [n for n, a in ARMS.items() if a["tier"] <= tier_max and (only is None or n in only)]
    status = {}
    for n in names:
        for t in tasks:
            try:
                status[(n, t)] = lb_job(n, t)
            except Blocked as e:
                status[(n, t)] = "BLOCKED"
                if t == tasks[0]:
                    print(f"BLOCKED {n}: {e}")
    print(f"\n{'arm':22s}" + "".join(f"{t:>13s}" for t in tasks))
    for n in names:
        print(f"{n:22s}" + "".join(f"{status[(n, t)]:>13s}" for t in tasks))
    return status

## 6. Experiment 1: matched-state accuracy of the completion term

**1a (runnable).** One collector run per task on the shipped method, which drives the trajectory. Every quantity below
compares the fill for unread windows (U) with the exact int2 read for read windows (R) **within the same
(step, layer, query head)**. So the step's normaliser (`Z_A · (1 + Σfill)` against the full-KV `Z`) cancels.
The collector's own "fill calibration" row compares `cache_step_mass` with `full_mass` directly and keeps that normaliser
as a per-step bias.

With `r(w) = log cache_step_mass(w) − log full_mass(w)`:

| Statistic | Meaning |
|---|---|
| `unread_mass` | full-KV attention mass on unread int2 windows: what the completion stands in for |
| `bias` (unweighted / mass-weighted) | mean `r` on U minus mean `r` on R: how far the fill's scale is off relative to reading. This is the selection bias, because `δ` is fitted on R and applied to U |
| `E_logm` (unweighted / mass-weighted) | mean `|r − r̄_R|` over U, the reviewer's log-mass error. The unweighted form is dominated by near-empty windows; the weighted form is the one that moves outputs |
| `tot_err` | `log(Σ_U fill / Σ_U true) − log(Σ_R read / Σ_R true)`, the error on the total unread mass |
| `sd_U` | residual spread on U. Its median is `σ̂_U`, the unit of the Experiment 4B noise levels |
| `read_gate_err` | relative L2 output change from reading every held window to the gated + completed read (the collector's `read_gate` rung) |

Statistics are taken per query head first: the median over steps, then the median, p90 and worst head
(`CLAUDE.md`: never pool across query heads before normalising per head). Windows below `FLOOR` = 1e-5 mass on either side
are left out of the unweighted statistics because fp16 storage is too coarse there. **Caveats:** the reference is the
original fp attention, so `r` on R includes the int2 error, and the bias reads as *fill versus reading*. The prompts are the first
`EXP1_PREFILL` tokens of each task's contexts plus a greedy continuation, not the task question.

**1b (blocked on this commit):** counterfactual outputs per variant, the mass oracle and `m*(u)` over `Z_A`.

In [ ]:
def exp1_job(task):
    out = ROOT / "exp1" / f"{task}_p{EXP1_PREFILL}_n{EXP1_GEN}_k{EXP1_PROMPTS}.zip"
    if out.exists():
        return out
    local = RUNS / "exp1" / out.name
    local.parent.mkdir(parents=True, exist_ok=True)
    local.unlink(missing_ok=True)
    run(["bash", "scripts/collect_observations.sh", "--model-path", MODEL_DIR, "--dataset", task,
         "--window-size", OPERATING_POINT["window_size"], "--cache-budget", OPERATING_POINT["cache_budget"],
         "--gate-ratio", GATE, "--quant-ratio", OPERATING_POINT["quant_ratio"],
         "--quant-budget-mode", OPERATING_POINT["quant_budget_mode"],
         "--prefill", EXP1_PREFILL, "--gen", EXP1_GEN, "--num-samples", EXP1_PROMPTS,
         "--out", local], f"exp1_{task}")
    meta = json.loads(zipfile.ZipFile(local).read("meta.json"))
    v = (meta.get("read_gate") or {}).get("verdict")
    if v != "gated":
        _refuse([local], f"exp1_{task}", [f"collector read_gate verdict {v!r}, need 'gated'"])
    shutil.copy(local, out)
    return out

EXP1_ZIPS = {t: exp1_job(t) for t in TASKS}

In [ ]:
import numpy as np, pandas as pd

FLOOR = 1e-5
EXP1_KEYS = ("tier", "gate_open", "gate_fired", "cache_step_mass", "full_mass", "out_step_err")
STATS = ("unread_mass", "bias_unw", "bias_w", "E_unw", "E_w", "tot_err", "sd_U", "read_gate_err")

def iter_zip(path, keys):
    with zipfile.ZipFile(path) as zf:
        meta = json.loads(zf.read("meta.json"))
        for s in meta["samples"]:
            with np.load(io.BytesIO(zf.read(s["file"])), allow_pickle=False) as z:
                yield {k: z[k] for k in keys}

def _head_step_stats(tier_tl, go_tl, csm_tl, fm_tl, rg_tl, rep):
    """One (step, layer): arrays over query heads. NaN where a head has no read or no unread window."""
    q = (tier_tl == 1)[None, :]
    op = np.repeat(go_tl, rep, axis=0)
    R, U = q & op, q & ~op
    c, f = csm_tl.astype(np.float64), fm_tl.astype(np.float64)
    ok = (c >= FLOOR) & (f >= FLOOR)
    r = np.log(np.maximum(c, FLOOR)) - np.log(np.maximum(f, FLOOR))
    Rk, Uk = R & ok, U & ok
    nR, nU = Rk.sum(1), Uk.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        mR = np.where(nR > 0, (r * Rk).sum(1) / nR, np.nan)
        mU = np.where(nU > 0, (r * Uk).sum(1) / nU, np.nan)
        wR, wU = f * Rk, f * Uk
        mRw = (wR * r).sum(1) / wR.sum(1)
        mUw = (wU * r).sum(1) / wU.sum(1)
        E_unw = np.where(nU > 0, (np.abs(r - mR[:, None]) * Uk).sum(1) / nU, np.nan)
        E_w = (wU * np.abs(r - mRw[:, None])).sum(1) / wU.sum(1)
        dev = np.where(Uk, r - mU[:, None], 0.0)
        sd_U = np.where(nU >= 4, np.sqrt((dev ** 2).sum(1) / np.maximum(nU - 1, 1)), np.nan)
        cU, fU = (c * U).sum(1), (f * U).sum(1)
        cR, fR = (c * R).sum(1), (f * R).sum(1)
        tot = np.log(cU / fU) - np.log(cR / fR)
    valid = (nR > 0) & (nU > 0)
    out = np.stack([fU, mU - mR, mUw - mRw, E_unw, E_w, tot, sd_U, rg_tl.astype(np.float64)], 1)
    out[~valid] = np.nan
    out[~np.isfinite(out)] = np.nan
    return out                                               # [H, len(STATS)]

def exp1a(zp):
    blocks = []
    for si, s in enumerate(iter_zip(zp, EXP1_KEYS)):
        tier, go, gf = s["tier"], s["gate_open"], s["gate_fired"]
        csm, fm, rg = s["cache_step_mass"], s["full_mass"], s["out_step_err"][..., 3]
        T, L, H, _ = csm.shape
        rep = H // go.shape[2]
        heads = np.arange(H, dtype=np.float64)[:, None]
        for t in range(1, T):
            for l in np.flatnonzero(gf[t]):
                st = _head_step_stats(tier[t, l], go[t, l], csm[t, l], fm[t, l], rg[t, l], rep)
                ids = np.broadcast_to(np.array([si, t, l], dtype=np.float64), (H, 3))
                blocks.append(np.hstack([ids, heads, st]))
    df = pd.DataFrame(np.vstack(blocks), columns=["sample", "step", "layer", "head", *STATS])
    return df.astype({"sample": int, "step": int, "layer": int, "head": int})

def summarise(df):
    per_head = df.groupby(["layer", "head"])[list(STATS)].median()
    worst = {k: (per_head[k].abs().max() if k.startswith(("bias", "tot")) else per_head[k].max())
             for k in STATS}
    return pd.DataFrame({"median head": per_head.median(), "p90 head": per_head.quantile(0.9),
                         "worst head": pd.Series(worst)})

E1 = {t: exp1a(z) for t, z in EXP1_ZIPS.items()}
allsteps = pd.concat(E1.values(), ignore_index=True)
for t, df in E1.items():
    print(f"\n=== {t}: {df['step'].nunique()} steps x {df['layer'].nunique()} layers x {df['head'].nunique()} heads")
    print(summarise(df).round(4).to_string())

sigma_hat = float(np.nanmedian(allsteps["sd_U"]))
c_hat = float(np.exp(np.nanmedian(allsteps["tot_err"])))
SIGMA_FILE.write_text(json.dumps({
    "sigma_hat_U": sigma_hat, "c_hat": c_hat, "floor": FLOOR, "sha": SHA,
    "per_task": {t: {"sigma_hat_U": float(np.nanmedian(d["sd_U"])),
                     "c_hat": float(np.exp(np.nanmedian(d["tot_err"]))),
                     "unread_mass_median": float(np.nanmedian(d["unread_mass"]))} for t, d in E1.items()},
}, indent=2))
print(f"\nsigma_hat_U = {sigma_hat:.3f} nats (Experiment 4B unit)   "
      f"c_hat = {c_hat:.3f} (measured common-scale error of the fill vs reading; marked on 4A)")
pd.concat({t: summarise(d) for t, d in E1.items()}).to_csv(ROOT / "results" / "exp1a_summary.csv")

In [ ]:
if not READY["exp1b"]:
    print("Experiment 1b BLOCKED on this commit: the collector has no completion rungs. Missing: per-variant "
          "counterfactual outputs on the same state (off / uncal / cal / scale / noise), the mass oracle "
          "(m*, stored centroid), Z_A-normalised m*(u) and nu*_u, and the mass-vs-centroid error split. "
          "This cell will be filled in with that collector change.")

## 7. Experiments 2–4: the LongBench grid

Tier 1 runs `cal`, `cal_repeat`, the ceiling, `off`, `uncal`, `scale0.5`, `scale2` and `noise1x` × 3 seeds. Tier 2 adds the
other scales, `noise2x` and `noise4x`, the ρ = 0.10 stress pair and the coupled arms. On this commit, every knob arm prints `BLOCKED` and does not run.

In [ ]:
STATUS = run_grid(TIER)

In [ ]:
if RUN_GSM8K and TIER >= 2:
    for n in GSM8K_ARMS:
        try:
            print(n, gsm8k_job(n))
        except Blocked as e:
            print(f"BLOCKED {n}: {e}")

## 8. Scoring and analysis

* Per-example scores use the repo's own LongBench metrics (`longbench_scoring.METRIC_FN_REGISTRY` and its first-line rule).
* Task scores follow THUDM: a null (OOM) prediction scores 0.
* Paired deltas use the examples that are non-null in **both** arms. They come with a 95% bootstrap CI and a
  changed-example rate, which should be read against `cal_repeat`.
* Every sidecar is re-verified here, and a file that fails is not scored.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
from modules.evaluation.longbench_scoring import METRIC_FN_REGISTRY, _FIRST_LINE_DATASETS
D2M = json.loads((REPO / "data/longbench_configs/dataset2metric.json").read_text())

def example_scores(jsonl, task):
    fn, out = METRIC_FN_REGISTRY[D2M[task]], {}
    for line in open(jsonl, encoding="utf-8"):
        if not line.strip():
            continue
        ex = json.loads(line)
        pred = ex.get("pred")
        if pred is None:
            out[ex["_id"]] = None
            continue
        if task in _FIRST_LINE_DATASETS:
            pred = pred.lstrip("\n").split("\n")[0]
        out[ex["_id"]] = 100.0 * max((fn(pred, gt, all_classes=ex.get("all_classes") or [])
                                      for gt in ex.get("answers", [])), default=0.0)
    return out

SCORES, SKIPPED = {}, []
for name, a in ARMS.items():
    for t in TASKS:
        d = ROOT / "longbench" / name
        if not (d / f"{t}.meta.json").exists():
            continue
        try:
            comp = resolved_comp(a)
        except Blocked:
            continue
        errs = verify(json.loads((d / f"{t}.meta.json").read_text()), a, comp)
        if errs:
            SKIPPED.append((name, t, errs))
            continue
        SCORES[(name, t)] = example_scores(d / f"{t}.jsonl", t)
for s in SKIPPED:
    print("NOT SCORED", s)

rng = np.random.default_rng(0)
def paired(x, y):
    ids = sorted(i for i in set(x) & set(y) if x[i] is not None and y[i] is not None)
    return np.array([x[i] for i in ids]), np.array([y[i] for i in ids])

def boot_dist(d):
    idx = rng.integers(0, len(d), (BOOTSTRAP_ITERS, len(d)))
    return d[idx].mean(1)

# THUDM-style task scores
arms_present = [n for n in ARMS if any((n, t) in SCORES for t in TASKS)]
tab = pd.DataFrame({t: {n: (np.mean([v if v is not None else 0.0 for v in SCORES[(n, t)].values()])
                            if (n, t) in SCORES else np.nan) for n in arms_present} for t in TASKS})
tab["average"] = tab[TASKS].mean(1)
print("Task scores (THUDM convention):")
print(tab.round(2).to_string())

# paired deltas vs cal
rows = []
for n in arms_present:
    if n == "cal" or not all((n, t) in SCORES and ("cal", t) in SCORES for t in TASKS):
        continue
    dists, means, row = [], [], {"arm": n, "exp": ARMS[n]["exp"]}
    for t in TASKS:
        x, y = paired(SCORES[(n, t)], SCORES[("cal", t)])
        d = x - y
        dists.append(boot_dist(d))
        means.append(d.mean())
        lo, hi = np.percentile(dists[-1], [2.5, 97.5])
        row[t] = f"{d.mean():+.2f} [{lo:+.2f},{hi:+.2f}]"
        row[f"{t}_changed"] = float((np.abs(d) > 1e-9).mean())
    # the average over tasks: each task resampled within itself, then the task means averaged
    row["avg_delta"] = float(np.mean(means))
    row["avg_lo"], row["avg_hi"] = np.percentile(np.mean(dists, 0), [2.5, 97.5])
    row["changed"] = float(np.mean([row[f"{t}_changed"] for t in TASKS]))
    rows.append(row)
DELTA = pd.DataFrame(rows).set_index("arm") if rows else pd.DataFrame()
if len(DELTA):
    print("\nPaired delta vs cal, points [95% CI]; 'changed' = share of examples whose score moved "
          "(compare with cal_repeat):")
    print(DELTA[[*TASKS, "avg_delta", "avg_lo", "avg_hi", "changed"]].round(3).to_string())
    DELTA.to_csv(ROOT / "results" / "paired_deltas.csv")
tab.to_csv(ROOT / "results" / "task_scores.csv")

# Experiment 2: gap closed = (cal - off) / (ceiling - off)
if all(("off", t) in SCORES and ("ceiling_g1.0", t) in SCORES for t in TASKS):
    num, den = [], []
    for t in TASKS:
        ids = sorted(set.intersection(*[{i for i, v in SCORES[(n, t)].items() if v is not None}
                                        for n in ("cal", "off", "ceiling_g1.0")]))
        c = np.array([SCORES[("cal", t)][i] for i in ids])
        o = np.array([SCORES[("off", t)][i] for i in ids])
        g = np.array([SCORES[("ceiling_g1.0", t)][i] for i in ids])
        idx = rng.integers(0, len(ids), (BOOTSTRAP_ITERS, len(ids)))
        num.append((c - o)[idx].mean(1))
        den.append((g - o)[idx].mean(1))
    N, D = np.mean(num, 0), np.mean(den, 0)
    if np.percentile(D, 2.5) > 0:
        gc = N / D
        print(f"\nExperiment 2 gap closed by completion: {np.median(gc):.2f} "
              f"[{np.percentile(gc, 2.5):.2f}, {np.percentile(gc, 97.5):.2f}]")
    else:
        print("\nExperiment 2 gap closed: undefined, because the ceiling does not beat 'off' significantly.")

# GSM8K (tier 2): paired accuracy deltas vs cal, same rules
if RUN_GSM8K and (ROOT / "gsm8k" / "cal" / "meta.json").exists():
    from modules.evaluation.gsm8k_scoring import gsm8k_score
    def gsm_scores(d):
        out = {}
        for line in open(d / "predictions.jsonl", encoding="utf-8"):
            if line.strip():
                ex = json.loads(line)
                out[ex["id"]] = None if ex.get("pred") is None else 100.0 * gsm8k_score(ex["pred"], ex["answer"])
        return out
    G = {}
    for n in GSM8K_ARMS:
        d = ROOT / "gsm8k" / n
        if (d / "meta.json").exists():
            try:
                comp = resolved_comp(ARMS[n])
            except Blocked:
                continue
            if not verify(json.loads((d / "meta.json").read_text()), ARMS[n], comp):
                G[n] = gsm_scores(d)
    for n, s in G.items():
        acc = np.mean([v or 0.0 for v in s.values()])
        if n == "cal":
            print(f"\nGSM8K cal accuracy {acc:.2f}")
            continue
        x, y = paired(s, G["cal"])
        lo, hi = np.percentile(boot_dist(x - y), [2.5, 97.5])
        print(f"GSM8K {n:12s} acc {acc:.2f}   Δ vs cal {np.mean(x - y):+.2f} [{lo:+.2f}, {hi:+.2f}]")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#52514e",
                     "axes.labelcolor": "#0b0b0b", "xtick.color": "#52514e", "ytick.color": "#52514e",
                     "axes.grid": True, "grid.color": "#e6e5e1", "grid.linewidth": 0.8,
                     "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb"})
BLUE, MUTED = "#2a78d6", "#52514e"
cal_noise = DELTA.loc["cal_repeat", "avg_delta"] if "cal_repeat" in DELTA.index else None

def _band(ax):
    ax.axhline(0, color=MUTED, lw=1, zorder=1)
    if cal_noise is not None:
        ax.axhspan(-abs(cal_noise), abs(cal_noise), color="#e6e5e1", lw=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# 4A: common scale (c = 0 is Experiment 2's 'off', drawn at the left edge)
ax = axes[0]; _band(ax)
pts = [(c, f"scale{c:g}") for c in (0.5, 0.8, 1.25, 2.0, 4.0)]
xs, ys, lo, hi = [0.0], [0.0], [0.0], [0.0]
for c, n in pts:
    if n in DELTA.index:
        xs.append(np.log2(c)); ys.append(DELTA.loc[n, "avg_delta"])
        lo.append(DELTA.loc[n, "avg_delta"] - DELTA.loc[n, "avg_lo"]); hi.append(DELTA.loc[n, "avg_hi"] - DELTA.loc[n, "avg_delta"])
order = np.argsort(xs)
ax.errorbar(np.array(xs)[order], np.array(ys)[order], yerr=[np.array(lo)[order], np.array(hi)[order]],
            color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3)
if "off" in DELTA.index:
    x0 = min(xs) - 1.2
    ax.errorbar([x0], [DELTA.loc["off", "avg_delta"]],
                yerr=[[DELTA.loc["off", "avg_delta"] - DELTA.loc["off", "avg_lo"]],
                      [DELTA.loc["off", "avg_hi"] - DELTA.loc["off", "avg_delta"]]],
                color=BLUE, marker="s", ms=8, capsize=3, lw=0, elinewidth=2)
    ax.annotate("off (c = 0)", (x0, DELTA.loc["off", "avg_delta"]), textcoords="offset points",
                xytext=(8, 0), va="center", color=MUTED)
if SIGMA_FILE.exists():
    ch = json.loads(SIGMA_FILE.read_text())["c_hat"]
    ax.axvline(np.log2(ch), color=MUTED, ls="--", lw=1, zorder=1)
    ax.annotate(f"measured fill error\nc = {ch:.2f}", (np.log2(ch), 1), xycoords=("data", "axes fraction"),
                textcoords="offset points", xytext=(-4, -4), ha="right", va="top", color=MUTED, fontsize=9)
ax.set_xlabel("log2 c  (common scale on unread-window mass)")
ax.set_ylabel("Δ LongBench average vs cal (points)")
ax.set_title("4A: common-scale mass error", loc="left")

# 4B: total-preserving redistribution noise
ax = axes[1]; _band(ax)
xs, ys, lo, hi = [0], [0.0], [0.0], [0.0]
for m in (1, 2, 4):
    v = [DELTA.loc[f"noise{m}x_s{s}", "avg_delta"] for s in range(3) if f"noise{m}x_s{s}" in DELTA.index]
    if v:
        xs.append(m); ys.append(np.mean(v)); lo.append(np.mean(v) - min(v)); hi.append(max(v) - np.mean(v))
ax.errorbar(xs, ys, yerr=[lo, hi], color=BLUE, lw=2, marker="o", ms=8, capsize=3, zorder=3)
ax.axvline(1, color=MUTED, ls="--", lw=1, zorder=1)
ax.annotate("measured residual\nspread " + r"$\hat{\sigma}_U$", (1, 1), xycoords=("data", "axes fraction"),
            textcoords="offset points", xytext=(-4, -4), ha="right", va="top", color=MUTED, fontsize=9)
ax.set_xlabel(r"$\sigma\,/\,\hat{\sigma}_U$" + "  (bars: min–max over 3 seeds)")
ax.set_title("4B: redistribution noise, total preserved", loc="left")
fig.text(0.01, -0.03, "Grey band: |cal_repeat − cal|, the run-to-run noise floor. Empty panels: arms blocked on this commit.",
         color=MUTED, fontsize=9)
fig.tight_layout()
fig.savefig(ROOT / "results" / "exp4_sensitivity.png", dpi=200, bbox_inches="tight")
plt.show()

## 9. Rebuttal table and run status

In [ ]:
def cell(n, t):
    return tab.loc[n, t] if n in tab.index and not np.isnan(tab.loc[n, t]) else None

label = {"cal": "Calibrated completion (QEvict)", "off": "Without completion", "uncal": "Uncalibrated completion",
         "scale0.8": "Perturbed masses, 0.8x", "scale1.25": "Perturbed masses, 1.25x",
         "scale0.5": "Perturbed masses, 0.5x", "scale2": "Perturbed masses, 2x", "scale4": "Perturbed masses, 4x",
         "ceiling_g1.0": "All windows read (rho = 1.0)", "cal_repeat": "Calibrated, repeat (noise floor)"}
lines = ["| Configuration | " + " | ".join(TASKS) + " | Average | Δ vs cal [95% CI] |",
         "|---|" + "---|" * (len(TASKS) + 2)]
for n, lab in label.items():
    vals = [cell(n, t) for t in TASKS]
    if all(v is None for v in vals):
        lines.append(f"| {lab} | " + " | ".join("—" for _ in TASKS) + " | — | "
                     + ("blocked on this commit |" if ARMS[n]["comp"] is not None and not READY["knob"] else "not run |"))
        continue
    avg = np.mean([v for v in vals if v is not None]) if all(v is not None for v in vals) else None
    d = (f"{DELTA.loc[n, 'avg_delta']:+.2f} [{DELTA.loc[n, 'avg_lo']:+.2f}, {DELTA.loc[n, 'avg_hi']:+.2f}]"
         if n in DELTA.index else "—")
    lines.append(f"| {lab} | " + " | ".join("—" if v is None else f"{v:.2f}" for v in vals)
                 + f" | {'—' if avg is None else f'{avg:.2f}'} | {d} |")
md_table = "\n".join(lines)
(ROOT / "results" / "rebuttal_table.md").write_text(md_table + f"\n\ncommit {SHA}\n")
print(md_table)
print(f"\nAll results: {ROOT / 'results'}   commit {SHA[:10]}   knob present: {READY['knob']}")